# 07 — Path integrals for general computation

One idea — an **expectation over trajectories**, each weighted by `exp(−action)` — reused across domains:

| rung | domain | what you do |
|---|---|---|
| 1 | **control** (MPPI) | weight sampled rollouts, act |
| 2 | **density propagation** (PATHINT) | fold a probability kernel |
| 3 | **statistical mechanics / inference** (SMNI, CMI, FEP) | action → momenta |

No path-integral background assumed. Each rung is the *same* construct in a new costume.

## Rung 1 — MPPI: path integrals as control
Sample many action sequences, roll them out, weight each by `exp(−cost/λ)`, and command the weighted-average action. That weighting *is* a path integral over control trajectories.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

def mppi(x0=0.0, goal=1.0, H=20, K=300, lam=0.1, steps=40):
    u = np.zeros(H); x = x0; traj = [x]
    for _ in range(steps):
        noise = rng.normal(0, 0.3, (K, H))
        xx = x + np.cumsum(u[None, :] + noise, axis=1)   # rollouts
        costs = ((xx - goal) ** 2).sum(axis=1)
        w = np.exp(-(costs - costs.min()) / lam); w /= w.sum()
        u = u + w @ noise                                 # path-integral update
        x = x + u[0]; traj.append(x); u = np.roll(u, -1); u[-1] = 0.0
    return np.array(traj)

traj = mppi()
print('MPPI final position:', round(float(traj[-1]), 3), '(goal = 1.0)')

MPPI final position: 0.835 (goal = 1.0)


## Rung 2 — PATHINT: propagate a *density*
Instead of choosing one action, fold a whole probability density forward through the short-time Gaussian kernel `T` (Ingber's PATHINT). We validate it against two analytic results: free diffusion spreads as `var = σ²·t`, and an Ornstein–Uhlenbeck drift relaxes to the stationary variance `σ²/(2k)`.

In [2]:
import jax.numpy as jnp
from qcccm.neuroai import pathint as pi

x = jnp.linspace(-6, 6, 401)
# free diffusion (no drift)
T = pi.build_transition_matrix(x, pi.linear_drift(x, 0.0), diffusion=1.0, dt=0.01)
free = pi.propagate(pi.delta_density(x, 0.0), T, 50)
_, var = pi.moments(free[-1], x)
print('free diffusion var :', round(float(var), 3), ' analytic σ²t =', 1.0*0.01*50)

# Ornstein-Uhlenbeck relaxation g(x) = -k x
k = 2.0
T2 = pi.build_transition_matrix(x, pi.linear_drift(x, -k), 1.0, 0.01)
ou = pi.propagate(pi.delta_density(x, 3.0), T2, 800)
m, v = pi.moments(ou[-1], x)
print('OU stationary mean :', round(float(m), 3))
print('OU stationary var  :', round(float(v), 3), ' analytic σ²/2k =', 1.0/(2*k))

free diffusion var : 0.5  analytic σ²t = 0.5
OU stationary mean : 0.0
OU stationary var  : 0.253  analytic σ²/2k = 0.25


In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
xg = np.array(x)
for i in [0, 40, 150, 799]:
    plt.plot(xg, np.array(ou[i]), label=f'step {i}')
plt.legend(); plt.xlabel('x'); plt.ylabel('P(x)')
plt.title('PATHINT: OU density relaxing to its stationary distribution')
plt.tight_layout()

## Rung 3 — SMNI / CMI (and qPATHINT next)
The SMNI Lagrangian `L = ½(Ṁ−g)ᵀΣ⁻¹(Ṁ−g)` defines the action; its **conjugate momenta** are the Canonical Momenta Indicators (CMI), `Π = Σ⁻¹(Ṁ−g)` — the same object the free-energy path integral uses. **qPATHINT** is rung 2 with a *complex* amplitude kernel (identical machinery, complex `T`) — the quantum step.

In [4]:
import jax
from qcccm.models import smni

M = jax.random.normal(jax.random.PRNGKey(0), (4, 3, 64))
d = smni.fit_linear_drift(M)
cmi = smni.canonical_momenta(M, d)
print('CMI shape', cmi.shape, '= conjugate momenta  Π = Σ⁻¹(Ṁ − g)')

CMI shape (4, 3, 64) = conjugate momenta  Π = Σ⁻¹(Ṁ − g)
